# Figure 2 — Synergy among a TF's own (direct) motif sites

Each direct motif site of a peak is knocked out alone and all sites together (`pipeline/04_direct_sites`, overlapping
sites merged), in the native peak (**WT**) and in a dinucleotide-shuffled peak that keeps only the direct sites
(**DiSh**). Predicted log counts, no control track, mean of 5 folds.

synergy = Σᵢ (ref − KOᵢ) − (ref − KOALL): > 0 cooperative, < 0 redundant.

Inputs: `data/examples/HepG2_FOXA1_ENCSR000BMO/` (steps 03, 04), the ENCODE models and bigBed of that experiment
(step 00), `data/self_epistasis_merged_direct_calc_wt.parquet` (step 05). The logo cells need a GPU.

In [ ]:
import sys
sys.path.insert(0, '../modules')        # run from notebooks/
import paths                            # data root, genome, repo locations (env BPNET_DATA, MISHA_HG38)
from plot_setup import fig_path         # editable PDF text; figures/ in the repository
%config InlineBackend.figure_format = 'retina'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymisha as pm
from pypalettes import load_cmap
from peaks import load_experiment, peak_interval, attribution_logo, peak_knockouts, loss_bars
from curves import self_synergy, per_tf, plot_epi_curves, plot_rank, HIGHLIGHT

pm.gsetroot(paths.MISHA_HG38)
teals = load_cmap('Teal', cmap_type='continuous')

## FOXA1 (HepG2, ENCSR000BMO): a synergistic and an additive peak

In [ ]:
foxa1 = load_experiment('HepG2/FOXA1_ENCSR000BMO', 'all')    # ALL_BPNET/HepG2/FOXA1_ENCSR000BMO

In [ ]:
self_epi = self_synergy(foxa1['direct_peaks'], foxa1['direct_sites'])     # peaks with >= 2 direct sites
selected = (self_epi[['peak_chrom', 'peak_start', 'peak_end', 'synergy_WT', 'synergy_DiSh', 'n_direct']]
            .rename(columns={'peak_chrom': 'chrom', 'peak_start': 'start', 'peak_end': 'end'}).reset_index())

# candidates: >= 2 direct sites, > 1 other motif, positive synergy in the native peak
out = (foxa1['epi']
       .merge(selected, on=['chrom', 'start', 'end'])
       .query('n_other > 1 & n_direct >= 2 & synergy_WT > 0')
       .sort_values('synergy_WT', ascending=False)
       .set_index('peak_name')
       [['chrom', 'start', 'end', 'rank', 'n_p0', 'n_other', 'interaction', 'synergy_WT', 'synergy_DiSh']])

SYNERGY_PEAK = 'HepG2_FOXA1_ENCSR000BMO_22769'     # out.iloc[0]: the strongest WT synergy (+0.85)
ADDITIVE_PEAK = 'HepG2_FOXA1_ENCSR000BMO_46012'    # 16th-smallest positive WT synergy (~0), with a DiSh synergy
out.loc[[SYNERGY_PEAK, ADDITIVE_PEAK]]

In [ ]:
interval = peak_interval(out.loc[SYNERGY_PEAK])
p = attribution_logo(interval, foxa1['hits'], foxa1['cfolds'])      # GPU
p.show()
p.save(fig_path('FOXA1_self_epi_synergy_seqlets.pdf'))

In [ ]:
levels, losses, synergy, pk = peak_knockouts(interval, foxa1['direct_sites'], foxa1['direct_peaks'])
print(pk.peak_name, {c: round(v, 3) for c, v in synergy.items()})
# left bar = single losses stacked, right bar = joint loss: left taller -> cooperative, right taller -> redundant
p = loss_bars(losses, teals)
p.show()
p.save(fig_path('FOXA1_self_epi_synergy_barplots.pdf'))

In [ ]:
interval = peak_interval(out.loc[ADDITIVE_PEAK])
p = attribution_logo(interval, foxa1['hits'], foxa1['cfolds'])      # GPU
p.show()
p.save(fig_path('FOXA1_self_epi_additivity_seqlets.pdf'))

In [ ]:
levels, losses, synergy, pk = peak_knockouts(interval, foxa1['direct_sites'], foxa1['direct_peaks'])
print(pk.peak_name, {c: round(v, 3) for c, v in synergy.items()})
p = loss_bars(losses, teals)
p.show()
p.save(fig_path('FOXA1_self_epi_additivity_barplot.pdf'))

## Generality: direct-site synergy across all TFs
Per experiment, the fraction of peaks (with ≥ 2 direct sites) whose WT synergy is ≥ t; one curve per TF = mean over
its experiments (SEM band for the highlighted TFs).

In [ ]:
calc_curves = pd.read_parquet(paths.DATA / 'self_epistasis_merged_direct_calc_wt.parquet')
per_tf_pos, per_tf_pos_sem = per_tf(calc_curves)
print(len(calc_curves), 'experiments,', len(per_tf_pos), 'TFs')

fig, ax = plt.subplots(1, figsize=(6, 6), constrained_layout=True)
plot_epi_curves(per_tf_pos, ax=ax, logy=True, sem=per_tf_pos_sem, highlight=HIGHLIGHT,
                ylabel='fraction of peaks with direct-site synergy ≥ t',
                xlabel='synergy threshold t (log counts)')
ax.set_title('direct-site synergy (WT) ≥ t')
plt.show()
fig.savefig(fig_path('curves_self_native.pdf'), dpi=300)

In [ ]:
fig, ranked = plot_rank(calc_curves, col='0.7', top_n=30, quantity='synergy')
plt.show()
fig.savefig(fig_path('epi07_rank_self_native.pdf'))